# LangGraph y Agentes

En el notebook anterior construimos sistemas RAG con LangChain: cargamos documentos, los dividimos en chunks, creamos embeddings y recuperamos contexto para responder preguntas. El flujo era **lineal y fijo**.

Ahora vamos a dar el siguiente paso: **¿qué pasa cuando el sistema necesita tomar decisiones por sí mismo?**

---

## ¿Qué es LangGraph?

LangGraph es una librería que permite construir flujos de trabajo con LLMs usando **grafos**.

Un grafo tiene:
- **Nodos**: pasos o acciones (llamar al LLM, ejecutar una herramienta...)
- **Aristas (edges)**: conexiones entre pasos, que pueden ser condicionales

Esto nos permite crear **bucles** y **bifurcaciones**: el sistema puede repetir pasos o tomar caminos distintos según lo que ocurra.

## ¿Qué es un Agente?

Un agente es un sistema donde el **LLM decide qué hacer** en cada momento:
- Qué herramienta usar
- Si necesita más información
- Cuándo tiene suficiente para responder

Lo iremos construyendo paso a paso, de lo más simple a lo más complejo.

In [ ]:
!{sys.executable} -m pip install langgraph
!{sys.executable} -m pip install langchain_openai
!{sys.executable} -m pip install langchain_core
!{sys.executable} -m pip install langchain-community
!{sys.executable} -m pip install faiss-cpu

In [ ]:
API_KEY = ""

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


loader = TextLoader("responsible_IA.txt")
documents = loader.load()
texts = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(documents)
embeddings = OpenAIEmbeddings(api_key=API_KEY)
vectorstore = FAISS.from_documents(texts, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0, api_key=API_KEY)
print(f"Vectorstore listo con {len(texts)} chunks")

In [ ]:
# Guardamos el vectorstore en disco
# FAISS crea una carpeta con dos ficheros:
#   index.faiss  → los vectores
#   index.pkl    → los documentos y metadatos
VECTORSTORE_PATH = "vectorstore_responsible_ia"

vectorstore.save_local(VECTORSTORE_PATH)
print(f"Vectorstore guardado en '{VECTORSTORE_PATH}'")

In [ ]:
# La próxima vez que ejecutemos el notebook no necesitamos recalcular nada
# Cargamos directamente desde disco, sin llamar a la API de embeddings
vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True  # FAISS usa pickle internamente, solo cargar ficheros propios
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vectorstore cargado desde disco")

---
## Paso 1 — Un solo nodo

El grafo más simple posible: `START → nodo → END`.

Vamos a construir un LLM que responde preguntas sin ningún RAG todavía. El objetivo es entender la mecánica de LangGraph antes de añadir complejidad.

En LangGraph el **estado** que pasa entre nodos cuando usamos `MessagesState` es simplemente una lista de mensajes, igual que en un chat normal.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage

# Un nodo es simplemente una función que recibe el estado y devuelve una actualización
# MessagesState contiene un campo "messages" que es una lista de mensajes
def call_llm(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph_1 = builder.compile()
print("Grafo compilado: START -> llm -> END")

In [ ]:
from IPython.display import Image
Image(graph_1.get_graph().draw_mermaid_png())

In [ ]:
result = graph_1.invoke({"messages": [HumanMessage("¿Qué es la IA responsable?")]})
print(result["messages"][-1].content)

---
## Paso 2 — Dos nodos: retrieve + generate (con mensajes)

Ahora añadimos RAG. Dos nodos en secuencia:

```
START -> retrieve -> generate -> END
```

Vamos a trabajar **solo con mensajes**, igual que en un chat real. El nodo `retrieve` añade un mensaje con los documentos encontrados, y el nodo `generate` los usa como contexto.

Fíjate en cómo los mensajes se van acumulando en el estado: cada nodo lee los mensajes anteriores y añade los suyos.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Nodo 1: recupera documentos e inyecta su contenido como un mensaje
def retrieve(state: MessagesState):
    question = state["messages"][-1].content
    docs = retriever.invoke(question)
    # Empaquetamos los documentos como un AIMessage con nombre
    # para que el siguiente nodo los pueda identificar en el estado
    docs_content = "\n\n---\n\n".join(doc.page_content for doc in docs)
    return {"messages": [AIMessage(content=docs_content, name="retrieve")]}


# Nodo 2: genera la respuesta usando el contexto que está en el estado
def generate(state: MessagesState):
    context = state["messages"][-1].content   # mensaje del nodo retrieve
    question = state["messages"][0].content   # primer HumanMessage

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Usa el siguiente contexto para responder.\n\nContexto:\n{context}"),
        ("human", "{question}"),
    ])
    response = (prompt | llm).invoke({"context": context, "question": question})
    return {"messages": [response]}


builder2 = StateGraph(MessagesState)
builder2.add_node("retrieve", retrieve)
builder2.add_node("generate", generate)
builder2.add_edge(START, "retrieve")
builder2.add_edge("retrieve", "generate")
builder2.add_edge("generate", END)

graph_2 = builder2.compile()
print("Grafo compilado: START -> retrieve -> generate -> END")

In [ ]:
Image(graph_2.get_graph().draw_mermaid_png())

In [ ]:
result2 = graph_2.invoke({"messages": [HumanMessage("¿Cuáles son los principios de la IA responsable?")]})

print("RESPUESTA:")
print(result2["messages"][-1].content)
print("\n--- Todos los mensajes en el estado ---")
for i, msg in enumerate(result2["messages"]):
    nombre = getattr(msg, 'name', type(msg).__name__)
    print(f"[{i}] {nombre}: {str(msg.content)[:80]}...")

# EJERCICIO 1
Modifica el grafo para que el LLM responda siempre en inglés,independientemente del idioma de la pregunta.


In [ ]:
#Haz aquí tu codigo

# EJERCICIO 2
Añade un tercer nodo "summarize" al final del grafo que resuma la respuesta de generate en una sola frase. El flujo debe quedar: START -> retrieve -> generate -> summarize -> END

In [ ]:
#Haz aquí tu codigo

---
## Paso 3 — Lo mismo pero con Estado tipado

Hasta ahora usamos `MessagesState`, que solo tiene una lista de mensajes. Pero en muchos casos nos interesa un **estado más explícito y estructurado**: campos con nombres claros para cada dato que viaja por el grafo.

```
START -> retrieve -> generate -> END
```

El flujo es idéntico al anterior, pero ahora el estado tiene campos propios: `question`, `documents`, `answer`. Esto hace el código más legible y fácil de depurar.

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document

# Estado personalizado: cada campo tiene un nombre y tipo claros
class RAGState(TypedDict):
    question: str
    documents: List[Document]
    answer: str


def retrieve_v2(state: RAGState):
    print(f"  [retrieve] Buscando: '{state['question']}'")
    docs = retriever.invoke(state["question"])
    print(f"  [retrieve] {len(docs)} documentos encontrados")
    return {"documents": docs}


def generate_v2(state: RAGState):
    print(f"  [generate] Generando respuesta...")
    context = "\n\n".join(doc.page_content for doc in state["documents"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Usa el siguiente contexto para responder.\n\nContexto:\n{context}"),
        ("human", "{question}"),
    ])
    response = (prompt | llm).invoke({"context": context, "question": state["question"]})
    return {"answer": response.content}


builder3 = StateGraph(RAGState)
builder3.add_node("retrieve", retrieve_v2)
builder3.add_node("generate", generate_v2)
builder3.add_edge(START, "retrieve")
builder3.add_edge("retrieve", "generate")
builder3.add_edge("generate", END)

graph_3 = builder3.compile()
print("Grafo compilado con estado tipado")

In [ ]:
Image(graph_3.get_graph().draw_mermaid_png())

In [ ]:
result3 = graph_3.invoke({
    "question": "¿Cuáles son los principios de la IA responsable?",
    "documents": [],
    "answer": ""
})

print("RESPUESTA:")
print(result3["answer"])

### MessagesState vs Estado tipado

| | `MessagesState` | Estado tipado (`TypedDict`) |
|---|---|---|
| **Cómo pasa la info** | Todo son mensajes en una lista | Campos con nombre explícito |
| **Legibilidad** | El estado imita un chat | Cada dato tiene su lugar claro |
| **Cuándo usarlo** | Agentes conversacionales, chatbots | Pipelines con datos estructurados |

> A partir del siguiente paso volvemos a `MessagesState` porque los agentes con herramientas funcionan de forma nativa con mensajes: el LLM devuelve un mensaje con la herramienta a invocar, y el resultado de la herramienta se añade como otro mensaje.

# EJERCICIO 3
Añade un campo "num_docs" al RAGState que guarde cuántos documentos se han recuperado, e imprímelo junto con la respuesta final.

In [34]:
#Haz aquí tu codigo

---
## Paso 4 — Tool Calling y ToolNode

Hasta ahora el flujo era fijo: siempre se recuperaban documentos, siempre se generaba respuesta.

Ahora el **LLM decide por sí mismo** si necesita buscar documentos o si puede responder directamente. Esto es la base de un agente.

### ¿Cómo funciona el Tool Calling?

Los modelos modernos pueden devolver, en vez de texto, una **llamada estructurada a una función**:

```
# En vez de: "Los principios son..."
# El LLM devuelve: {"tool": "context_search", "args": {"query": "principios IA"}}
```

`ToolNode` ejecuta automáticamente esa herramienta y añade el resultado al estado como un `ToolMessage`.

### El flujo ahora tiene una bifurcación:

```
                          si tool_call
START -> agent ---------------------► retrieve -> generate -> END
              \
               ──────────────────────────────────────────────► END
                          sin tool_call (responde directo)
```

`tools_condition` es una función de LangGraph que revisa si el último mensaje del agente tiene tool calls.

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import ToolMessage

# Creamos la herramienta de búsqueda a partir del retriever
# La descripción es importante: el LLM la lee para decidir cuándo usarla
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="context_search",
    description="Busca y devuelve documentos relevantes sobre IA responsable. "
                "Úsala cuando necesites información del documento para responder.",
)
tools = [retriever_tool]

# Enlazamos las herramientas al LLM
# Esto le dice al modelo qué funciones puede invocar
agent_model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0, api_key=API_KEY)
agent_model = agent_model.bind_tools(tools)

agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente experto en IA responsable. "
     "Cuando el usuario haga una pregunta sobre el documento, "
     "SIEMPRE usa la herramienta 'context_search' para buscar información relevante antes de responder. "
     "Solo responde directamente si la pregunta es general y no requiere consultar el documento."),
    MessagesPlaceholder(variable_name="messages"),
])

agent_chain = agent_prompt | agent_model

def agent(state: MessagesState):
    print(f"  [agent] Procesando {len(state['messages'])} mensajes")
    response = agent_chain.invoke({"messages": state["messages"]})
    if response.tool_calls:
        print(f"  [agent] -> Quiere usar: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print(f"  [agent] -> Responde directamente")
    return {"messages": [response]}



# Nodo generate: genera la respuesta final usando los documentos recuperados
def generate_from_tool(state: MessagesState):
    print(f"  [generate] Generando respuesta con contexto...")
    # El ToolMessage contiene los documentos que devolvió context_search
    last_tool_message = next(
        m for m in reversed(state["messages"]) if isinstance(m, ToolMessage)
    )
    docs = last_tool_message.content

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Usa el siguiente contexto para responder.\n\nContexto:\n{context}"),
        MessagesPlaceholder(variable_name="messages"),
    ])
    response = (prompt | llm).invoke({"context": docs, "messages": state["messages"]})
    return {"messages": [response]}


# ToolNode ejecuta automáticamente las herramientas que el agente haya elegido
retrieve_node = ToolNode(tools)

builder4 = StateGraph(MessagesState)
builder4.add_node("agent", agent)
builder4.add_node("retrieve", retrieve_node)
builder4.add_node("generate", generate_from_tool)

builder4.add_edge(START, "agent")

# tools_condition: si el agente hizo tool_calls -> "retrieve", si no -> END
builder4.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "retrieve", END: END},
)
builder4.add_edge("retrieve", "generate")
builder4.add_edge("generate", END)

graph_4 = builder4.compile()
print("Grafo con tool calling compilado")

In [ ]:
Image(graph_4.get_graph().draw_mermaid_png())

In [ ]:
def print_message_flow(result):
    """Muestra el flujo completo de mensajes que ha recorrido el grafo."""
    print("\n--- FLUJO DE MENSAJES ---")
    for i, msg in enumerate(result["messages"]):
        tipo = type(msg).__name__
        nombre = getattr(msg, "name", None)
        label = f"{tipo}" + (f" [{nombre}]" if nombre else "")

        # Contenido resumido
        contenido = msg.content
        if isinstance(contenido, list):          # tool_calls a veces vienen como lista
            contenido = str(contenido)
        contenido_corto = contenido[:120].replace("\n", " ") if contenido else ""

        # Tool calls (cuando el agente decide usar una herramienta)
        tool_info = ""
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
            tool_info = f"\n      🔧 tool_calls: {', '.join(calls)}"

        print(f"  [{i}] {label}")
        if contenido_corto:
            print(f"      {contenido_corto}...")
        if tool_info:
            print(tool_info)
    print("-------------------------\n")

In [ ]:
print("--- Pregunta sobre el documento ---")
result4a = graph_4.invoke({"messages": [HumanMessage("¿Qué dice el documento sobre transparencia en IA?")]})
print_message_flow(result4a)
print("RESPUESTA FINAL:", result4a["messages"][-1].content)

In [ ]:
print("--- Pregunta general ---")
result4b = graph_4.invoke({"messages": [HumanMessage("¿Qué es el machine learning? Responde en una frase.")]})
print_message_flow(result4b)
print("RESPUESTA FINAL:", result4b["messages"][-1].content)

---
## Paso 5 — Memoria y conversación real

Hasta ahora cada ejecución es independiente: el agente no recuerda nada de lo que se dijo antes.

LangGraph tiene un sistema de **checkpoints**: guarda el estado después de cada ejecución. Con esto podemos:
- Continuar una conversación en la siguiente llamada
- Tener múltiples conversaciones en paralelo (cada una con su `thread_id`)

```python
config = {"configurable": {"thread_id": "usuario_123"}}
```

El grafo es exactamente el mismo que antes. Solo cambia el `compile(checkpointer=memory)` y que pasamos `config` al invocar.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver guarda el estado en RAM
# En producción se usaría una base de datos (SqliteSaver, PostgresSaver...)
memory = MemorySaver()

# El grafo es idéntico al anterior, solo añadimos checkpointer
graph_5 = builder4.compile(checkpointer=memory)
print("Grafo con memoria compilado")

In [ ]:
Image(graph_5.get_graph().draw_mermaid_png())

In [ ]:
# El thread_id identifica una conversación.
# Cada vez que invocamos con el mismo thread_id, el agente recuerda el historial.
config_user1 = {"configurable": {"thread_id": "usuario_1"}}

def chat(question: str, config: dict):
    print(f"\nUSUARIO: {question}")
    result = graph_5.invoke(
        {"messages": [HumanMessage(question)]},
        config=config
    )
    answer = result["messages"][-1].content
    print(f"AGENTE:  {answer}")
    return answer


print("=" * 60)
print("CONVERSACION — usuario_1")
print("=" * 60)

chat("¿Qué dice el documento sobre transparencia en IA?", config_user1)
chat("¿Y sobre equidad?", config_user1)   # Recuerda que estamos hablando del documento
chat("Resume en una frase lo que me has dicho hasta ahora.", config_user1)

In [ ]:
# Diferente thread_id -> conversación completamente nueva, sin memoria de la anterior
config_user2 = {"configurable": {"thread_id": "usuario_2"}}

print("=" * 60)
print("NUEVA CONVERSACION — usuario_2")
print("=" * 60)

chat("¿Qué me has dicho antes sobre transparencia?", config_user2)

# EJERCICIO 4
Simula dos usuarios hablando a la vez con el mismo agente. 

- usuario_A pregunta sobre transparencia en IA.
- usuario_B pregunta sobre equidad en IA.

Luego cada uno pregunta "¿qué me has dicho antes?" y debe recibir solo su propio historial.

---
## Paso 6 — Evaluación de relevancia (con bucle)

Nuestro agente siempre genera una respuesta cuando recupera documentos. Pero ¿qué pasa si los documentos recuperados no son relevantes para la pregunta?

Añadimos un nodo `grade_documents` que evalúa los documentos antes de generar la respuesta:

```
                                        relevantes
START -> agent -> retrieve -> grade ─────────────► generate -> END
           ▲                    │
           │                    │ no relevantes
           │                    ▼
           └──────── rewrite_question
```

Si los documentos **no son relevantes**, en vez de responder con información incorrecta, el agente **reformula la pregunta** y vuelve a buscar. Aquí aparece el **bucle**: el grafo puede iterar.

In [ ]:
from typing import Literal

# Nodo: evalúa si los documentos recuperados son relevantes
def grade_documents(state: MessagesState):
    print("  [grade] Evaluando relevancia de los documentos...")

    question = state["messages"][0].content
    last_tool_message = next(
        m for m in reversed(state["messages"]) if isinstance(m, ToolMessage)
    )
    docs = last_tool_message.content

    grading_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Eres un evaluador. Responde UNICAMENTE 'relevante' o 'no_relevante'.\n"
         "¿Los documentos contienen información útil para responder la pregunta?"),
        ("human", "Pregunta: {question}\n\nDocumentos:\n{docs}"),
    ])

    response = (grading_prompt | llm).invoke({"question": question, "docs": docs[:1000]})
    es_relevante = "no_relevante" not in response.content.lower()
    grade = "relevante" if es_relevante else "no_relevante"

    print(f"  [grade] Resultado: {grade}")
    return {"messages": [AIMessage(content=grade, name="grade")]}


# Nodo: reformula la pregunta cuando los documentos no son relevantes
def rewrite_question(state: MessagesState):
    print("  [rewrite] Reformulando la pregunta...")
    question = state["messages"][0].content

    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", "Reformula la siguiente pregunta para mejorar la búsqueda semántica. "
                   "Hazla más específica y clara. Devuelve solo la pregunta reformulada."),
        ("human", "{question}"),
    ])
    response = (rewrite_prompt | llm).invoke({"question": question})
    print(f"  [rewrite] Nueva pregunta: {response.content}")
    return {"messages": [HumanMessage(content=response.content)]}


# Función de enrutamiento: lee el resultado de grade y decide el camino
def route_after_grade(state: MessagesState) -> Literal["generate", "rewrite_question"]:
    last_grade = next(
        m for m in reversed(state["messages"]) if isinstance(m, AIMessage) and m.name == "grade"
    )
    if "no_relevante" in last_grade.content:
        print("  [route] -> Documentos no relevantes, reformulando pregunta")
        return "rewrite_question"
    print("  [route] -> Documentos relevantes, generando respuesta")
    return "generate"


def agent_v2(state: MessagesState):
    print(f"  [agent] Procesando...")
    response = agent_chain.invoke(state["messages"])
    if response.tool_calls:
        print(f"  [agent] -> Busca: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print(f"  [agent] -> Responde directamente")
    return {"messages": [response]}

def generate_v3(state: MessagesState):
    print("  [generate] Generando respuesta final...")
    last_tool_message = next(
        m for m in reversed(state["messages"]) if isinstance(m, ToolMessage)
    )
    docs = last_tool_message.content
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Usa el siguiente contexto para responder.\n\nContexto:\n{context}"),
        MessagesPlaceholder(variable_name="messages"),
    ])
    response = (prompt | llm).invoke({"context": docs, "messages": state["messages"]})
    return {"messages": [response]}


builder6 = StateGraph(MessagesState)
builder6.add_node("agent", agent_v2)
builder6.add_node("retrieve", ToolNode(tools))
builder6.add_node("grade_documents", grade_documents)
builder6.add_node("generate", generate_v3)
builder6.add_node("rewrite_question", rewrite_question)

builder6.add_edge(START, "agent")
builder6.add_conditional_edges("agent", tools_condition, {"tools": "retrieve", END: END})
builder6.add_edge("retrieve", "grade_documents")
builder6.add_conditional_edges(
    "grade_documents",
    route_after_grade,
    {"generate": "generate", "rewrite_question": "rewrite_question"},
)
# El bucle: después de reformular, volvemos al agente
builder6.add_edge("rewrite_question", "agent")
builder6.add_edge("generate", END)

graph_6 = builder6.compile()
print("Grafo con evaluacion compilado")

In [ ]:
Image(graph_6.get_graph().draw_mermaid_png())

In [ ]:
print("=" * 60)
print("Pregunta relevante")
print("=" * 60)
result6a = graph_6.invoke({"messages": [HumanMessage("¿Cuáles son los principios de IA responsable?")]})
print("\nRESPUESTA:", result6a["messages"][-1].content)

In [ ]:
print("=" * 60)
print("Pregunta ambigua (puede necesitar reformulacion)")
print("=" * 60)
result6b = graph_6.invoke({"messages": [HumanMessage("¿qué dice sobre el tema?")]})
print("\nRESPUESTA:", result6b["messages"][-1].content)

---
## Resumen: la evolución de los grafos

| Paso | Estructura | Qué añade |
|------|-----------|----------|
| 1 | `START -> llm -> END` | Un solo nodo, flujo mínimo |
| 2 | `START -> retrieve -> generate -> END` | Dos nodos en secuencia, RAG con mensajes |
| 3 | Igual pero con `TypedDict` | Estado tipado con campos nombrados |
| 4 | `agent -> retrieve -> generate` con bifurcación | El LLM decide si buscar o no (tool calling) |
| 5 | Igual + `MemorySaver` + `thread_id` | Memoria conversacional entre llamadas |
| 6 | Añade `grade -> rewrite -> agent` | Bucle: el agente evalúa y reformula si hace falta |

### ¿Qué hace que el paso 4 en adelante sea un **agente**?

El LLM ya no solo genera texto: **toma decisiones** sobre el flujo de ejecución. Decide qué herramienta invocar, o si puede responder sin buscar. El grafo no lo obliga a seguir un camino fijo.

En el paso 6, además, el sistema puede **iterar**: si la primera búsqueda no fue útil, lo intenta de nuevo con una pregunta mejor. Eso es comportamiento de agente.

---
## Ejercicio

Añade una segunda herramienta al agente del paso 5 que devuelva la fecha actual. El agente debe elegir correctamente entre `context_search` y la nueva herramienta según la pregunta.

```python
from langchain_core.tools import tool
from datetime import datetime

@tool
def get_current_date() -> str:
    """Devuelve la fecha y hora actuales."""
    # TODO
    pass
```

Pruébalo con:
- `"¿Qué dice el documento sobre privacidad?"`
- `"¿Qué fecha es hoy?"`
- `"¿Cuál es la fecha de hoy y qué principios de IA responsable se mencionan?"` ← ¿usará las dos herramientas?

In [ ]:
# Tu solución aquí
